# Cleaning — Contoso profitability

The cleaning pipeline for the whole project.

```
sql/02_validation.sql         count every defect on the raw stg_ tables
notebooks/03_cleaning.ipynb   you are here: decide, clean, write out
sql/04_analysis.sql           Q1-Q15 on the clean tables
notebooks/05_statistics.ipynb Simpson's, confounders, bootstrap
```

pandas for this step because string normalisation, multi-format date parsing and value
mapping are all noticeably shorter here than in SQL, and the recovery arithmetic reads
better. The analysis stays in SQL, which is where window functions and cohort queries
belong.

Outputs:

| | |
|---|---|
| `data/clean/*.csv` | seven clean tables |
| SQL Server `dim_*` / `fact_sales` | same tables, for `sql/04_analysis.sql` |

Every decision below is recorded with its reason in section 3.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 40)

RAW   = Path("../data/raw")
CLEAN = Path("../data/clean")
CLEAN.mkdir(parents=True, exist_ok=True)

PLACEHOLDERS = {"", "NA", "N/A", "NULL", "Unknown", "unknown", "-", "?"}

def load(name):
    # dtype=str            -> nothing is silently coerced on read
    # keep_default_na=False -> "NA" and "NULL" stay as TEXT, not NaN.
    #   Without this, pandas turns them into NaN before I can count them,
    #   and the distinction between "empty" and "someone typed NA" is gone.
    return pd.read_csv(RAW / f"{name}.csv", dtype=str, keep_default_na=False)

fact = load("Fact_Sales")
cust = load("Dim_Customer")
prod = load("Dim_Product")
region = load("Dim_Region")
chan = load("Dim_Channel")
date = load("Dim_Date")
usrg = load("Dim_UserRegion")

print(f"fact {fact.shape}   cust {cust.shape}   prod {prod.shape}")

## 1. Profile

Three questions about every column before anything else: how many distinct values, how
many blanks, and does it cast to the type I expect.

In [ ]:
def profile(df, name):
    rows = []
    for c in df.columns:
        s = df[c].astype(str).str.strip()
        rows.append({
            "column": c,
            "distinct": s.nunique(),
            "blank_or_placeholder": int(s.isin(PLACEHOLDERS).sum()),
            "pct_missing": round(100 * s.isin(PLACEHOLDERS).mean(), 2),
            "untrimmed": int((df[c].astype(str) != s).sum()),
        })
    print(f"\n=== {name}  ({len(df):,} rows) ===")
    return pd.DataFrame(rows)

profile(fact, "Fact_Sales")

In [ ]:
profile(cust, "Dim_Customer")

In [ ]:
profile(prod, "Dim_Product")

Worth reading the `untrimmed` column. Leading and trailing spaces are invisible both
in a dataframe display and in SSMS, and they are why a join can silently return nothing.
This is the cheapest way to find them.

## 2. Let the data enumerate its own defects

The instinct is `df[df.ShippingDays.isin(['-', '?', 'Unknown'])]`, which only finds what
you already thought of. My first pass listed exactly those three and there are seven. The
better question is what will not convert to a number.

In [ ]:
bad = fact.loc[pd.to_numeric(fact["ShippingDays"], errors="coerce").isna(), "ShippingDays"]
print(bad.value_counts().to_string())
print(f"\ntotal non-numeric: {len(bad):,}  ({100*len(bad)/len(fact):.2f}%)")

`errors="coerce"` turns failures into `NaN` instead of raising. It is the pandas
equivalent of SQL's `TRY_CAST`.

In [ ]:
# Categorical variants: raw distinct vs. distinct after trim+upper
for df, col in [(fact,"OrderStatus"), (fact,"PaymentMethod"),
                (cust,"Gender"), (cust,"Segment"), (prod,"Category"), (prod,"Brand")]:
    raw = df[col].nunique()
    clean = df[col].str.strip().str.upper().nunique()
    print(f"{col:<16} raw={raw:>3}   after trim+upper={clean:>3}")

`Gender`: trim and upper takes it to 9 distinct values, not 2, because `M` and `MALE`
are still different strings. Some columns need a value map rather than a format fix.

`Brand`: `Bajaj` and `bajaj` are the same manufacturer, and trim alone leaves them
split.

In [ ]:
# Date shapes - classify BEFORE parsing. Counting tells you how many are
# broken; classifying tells you what to write the parser for.
import re
def shape(s):
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", s):      return "ISO yyyy-mm-dd"
    if re.fullmatch(r"\d{2}/\d{2}/\d{4}", s):      return "NN/NN/yyyy"
    if re.fullmatch(r"\d{2}-[A-Za-z]{3}-\d{4}", s): return "dd-MMM-yyyy"
    if re.fullmatch(r"\d{4}/\d{2}/\d{2}", s):      return "yyyy/mm/dd"
    return "other"

print(fact["OrderDate"].map(shape).value_counts().to_string())

In [ ]:
# Numeric validity, duplicate keys, orphan foreign keys
q  = pd.to_numeric(fact["Quantity"], errors="coerce")
dp = pd.to_numeric(fact["DiscountPercent"], errors="coerce")
print(f"Quantity = 0        {int((q==0).sum()):>5}")
print(f"Quantity < 0        {int((q<0).sum()):>5}")
print(f"Discount > 1        {int((dp>1).sum()):>5}")
print(f"\nDim_Customer rows {len(cust):,} / keys {cust.CustomerID.nunique():,}"
      f"  -> {len(cust)-cust.CustomerID.nunique()} duplicate keys")
print(f"Dim_Product  rows {len(prod):,} / keys {prod.ProductID.nunique():,}"
      f"  -> {len(prod)-prod.ProductID.nunique()} duplicate keys")
for key, dim in [("CustomerID", cust), ("ProductID", prod), ("RegionID", region)]:
    orph = ~fact[key].isin(set(dim[key]))
    print(f"orphan {key:<11} {int(orph.sum()):>5} rows / {fact.loc[orph,key].nunique()} ids")

## 3. Decisions

Each row is a choice that could have gone the other way, with the reason attached.

| Issue | Rows | Decision | Why |
|---|---:|---|---|
| `ShippingDays` placeholders (7 forms) | 4,713 | null + `IsShippingKnown` flag | do not invent a delivery time; the flag keeps coverage visible |
| `OrderDate` non-ISO (4 shapes) | 1,571 | parse each format explicitly | recoverable, and dropping 1% would bias by date |
| `PaymentMethod` missing | 1,571 | null | the sale is valid, only the tender is unknown |
| casing / whitespace variants | ~1.5% | map to canonical values | `M` and `MALE` need a lookup, not a trim |
| `Quantity = 0` | 8 | recover from the money columns | `Sales / (Price * (1-Disc))` gives the quantity exactly |
| `DiscountPercent > 1` | 8 | recover from the money columns | `1 - Sales / (Qty * Price)` gives the discount exactly |
| `Quantity < 0` | 8 | exclude | credit notes, money negated to match, so not recoverable as sales |
| duplicate dimension keys | 15 | dedupe identical, survivorship for conflicting | two different problems with opposite fixes |
| orphan foreign keys | 34 | map to `UNKNOWN` members | never drop silently, the revenue stays visible |
| dimension attribute blanks | 24 customer, 5 product | left null, never imputed | `CustomerName`, `Gender`, `JoinDate` (24 each) and `Brand`, `LaunchDate` (5 each). Nothing in the data implies them. Compare `City`/`State` (16) and `AgeGroup` (24), which were recovered because a lookup or derivation existed. Derive where you can, null where you cannot, do not impute |
| `UNKNOWN` member rows | 1 per dimension | carry nulls in unset attributes | it is a placeholder for orphan keys, not a real entity. `dim_customer` sets only `CustomerID`/`CustomerName`/`Segment`/`AgeGroup`, so its other nine columns are null by design. Filter it out of attribute-level visuals or "Unknown Customer" shows up in every Gender slicer |
| money-rule breaches | 1,049 | flag, do not correct | no way to tell which side is wrong. 700 pricing + 349 profit, no overlap |
| negative profit | 17,418 | keep, it is not an error | it is what the analysis is about |
| cancelled orders | 4,834 | excluded from money metrics | ₹6.02 Cr of revenue that never happened |

On the recoveries: an earlier version excluded all 24 bad-quantity and bad-discount rows
on the grounds that "the money was not recalculated". That was backwards. The money
columns are the intact part, which is exactly what makes 16 of them recoverable. Only the
8 negative-quantity credit notes had their money negated too, so only those are genuinely
lost.

Two of these counts sit one below `Issue_Manifest.xlsx`, and both differences are
explained rather than missed.

Money-rule breaches are 1,049 where the manifest says 1,050. `ORD-2025-008816-02` states
profit ₹1.26 where sales minus cost is ₹1.27. The gap is ₹0.01 and the flag's tolerance
is ₹0.05. That tolerance exists so float and rounding noise does not register as a
business-rule failure, and a ₹0.01 drift on a ₹668 line is that noise. So 1,049 of 1,050
detected, by design.

Negative profit is 17,418 where the manifest implies 17,425. Seven of the eight excluded
negative-quantity credit notes also carried negative profit, so they left with the
exclusion. 17,425 - 7 = 17,418.

Both are noted here because anyone who opens the manifest will find the difference, and
an unexplained one-row gap looks like a bug.

## 4. Clean the dimensions

Dedupe first. `Dim_Customer` has 10 duplicate keys and `Dim_Product` has 5. Merging a
fact table to a non-unique key multiplies rows, so revenue inflates and nothing errors.

In [ ]:
GENDER_MAP = {"M":"Male","MALE":"Male","F":"Female","FEMALE":"Female"}
CAT_MAP    = {c.upper(): c for c in
              ["Technology","Furniture","Office Supplies","Appliances","Accessories"]}
SEG_MAP    = {c.upper(): c for c in
              ["Consumer","Small Business","Corporate","Enterprise"]}
PAY_MAP    = {c.upper(): c for c in
              ["Credit Card","Debit Card","UPI","Cash","Net Banking","Wallet"]}
STATUS_MAP = {"COMPLETED":"Completed","RETURNED":"Returned",
              "CANCELLED":"Cancelled","CANCELED":"Cancelled"}

def clean_text(s, mapping=None):
    """Trim, null every placeholder form, optionally map to a canonical value."""
    s = s.astype(str).str.strip()
    s = s.mask(s.isin(PLACEHOLDERS))      # 'NA', '-', 'NULL' ... all -> NaN
    return s.str.upper().map(mapping) if mapping else s

`clean_text` nulls the whole placeholder set rather than just the empty string.
`NULLIF(x, '')` only catches the empty string, so `'NA'` and `'-'` survived into the
first version of this pipeline, and because they are not null the City/State recovery
below silently did nothing for those rows.

In [ ]:
# --- Dim_Product ---------------------------------------------------------
# Survivorship: prefer ProductStatus 'Active', then lowest ListPrice, then
# ProductName. ListPrice is cast to a NUMBER first - sorting it as text puts
# '9.99' after '10.00' and a different row wins.
# kind="stable" matters: the default sort is not stable, so tied rows could
# survive differently between runs, and differently from the SQL version.
prod["_lp"] = pd.to_numeric(prod["ListPrice"], errors="coerce")
p = (prod.sort_values(["ProductID","ProductStatus","_lp","ProductName"], kind="stable")
         .drop_duplicates("ProductID", keep="first").drop(columns="_lp").copy())

p["Category"]     = clean_text(p["Category"], CAT_MAP)
p["SubCategory"]  = clean_text(p["SubCategory"])
p["ProductName"]  = clean_text(p["ProductName"])
p["ListPrice"]    = pd.to_numeric(p["ListPrice"], errors="coerce")
p["StandardCost"] = pd.to_numeric(p["StandardCost"], errors="coerce")
p["LaunchDate"]   = pd.to_datetime(p["LaunchDate"], format="%Y-%m-%d", errors="coerce")

# Brand: one spelling per value - take the most frequent ('Bajaj', not 'bajaj')
b = clean_text(prod["Brand"]).dropna()
BRAND_MAP = b.groupby(b.str.upper()).agg(lambda s: s.value_counts().index[0]).to_dict()
p["Brand"] = clean_text(p["Brand"]).str.upper().map(BRAND_MAP)

# --- Dim_Customer --------------------------------------------------------
c = (cust.sort_values(["CustomerID","CustomerStatus","Segment","City","RegionID"],
                      kind="stable")
         .drop_duplicates("CustomerID", keep="first").copy())

c["CustomerName"] = clean_text(c["CustomerName"])
c["Gender"]       = clean_text(c["Gender"],  GENDER_MAP)
c["Segment"]      = clean_text(c["Segment"], SEG_MAP)
c["City"]         = clean_text(c["City"])
c["State"]        = clean_text(c["State"])
c["DateOfBirth"]  = pd.to_datetime(c["DateOfBirth"], format="%Y-%m-%d", errors="coerce")
c["JoinDate"]     = pd.to_datetime(c["JoinDate"],    format="%Y-%m-%d", errors="coerce")

# Age recomputed, not trusted - 5 source values are impossible (0, negative, >120).
# AgeGroup derived from it, so the 24 blanks disappear. Derive beats impute.
c["Age"] = ((pd.Timestamp("2025-12-31") - c["DateOfBirth"]).dt.days // 365).astype("Int64")
c["AgeGroup"] = pd.cut(c["Age"], [0,24,34,44,54,64,200],
                       labels=["18-24","25-34","35-44","45-54","55-64","65+"]).astype(object)
c.loc[c["DateOfBirth"].isna(), "AgeGroup"] = None   # never let null fall into '65+'

# City/State blanks recovered from the region - available, so not imputed
r_lookup = region.set_index("RegionID")[["City","State"]]
c["City"]  = c["City"].fillna(c["RegionID"].map(r_lookup["City"]))
c["State"] = c["State"].fillna(c["RegionID"].map(r_lookup["State"]))

print(f"products {len(prod)} -> {len(p)}    brands {p.Brand.nunique()}")
print(f"customers {len(cust)} -> {len(c)}   AgeGroup nulls {int(c.AgeGroup.isna().sum())}")

The `AgeGroup` null guard is load-bearing. Without the explicit
`c.loc[DateOfBirth.isna(), "AgeGroup"] = None`, a missing date of birth produces a null
age, which fails every `pd.cut` bin and lands in the last bucket. A customer with no
birth date would be reported as 65+. It does not bite on this data but it is wrong as
soon as the source changes.

## 5. Clean the fact table

In [ ]:
f = fact.copy()

# --- Dates: try each format, keep the first that parses -------------------
# Works because the slash formats only ever use days > 12, so exactly one
# interpretation is valid per row and no silent day/month swap is possible.
d = pd.to_datetime(f["OrderDate"], format="%Y-%m-%d", errors="coerce")
for fmt in ("%d/%m/%Y", "%m/%d/%Y", "%d-%b-%Y", "%Y/%m/%d"):
    d = d.fillna(pd.to_datetime(f["OrderDate"], format=fmt, errors="coerce"))
f["OrderDate"] = d
print("unparsed dates:", int(f["OrderDate"].isna().sum()), "(must be 0)")

# --- Types ----------------------------------------------------------------
f["Quantity"] = pd.to_numeric(f["Quantity"], errors="coerce").astype("Int64")
for col in ["UnitPrice","DiscountPercent","SalesAmount","CostAmount","ProfitAmount"]:
    f[col] = pd.to_numeric(f[col], errors="coerce")

f["ShippingDays"]    = pd.to_numeric(f["ShippingDays"], errors="coerce").astype("Int64")
f["IsShippingKnown"] = f["ShippingDays"].notna()
f["OrderStatus"]     = clean_text(f["OrderStatus"],   STATUS_MAP)
f["PaymentMethod"]   = clean_text(f["PaymentMethod"], PAY_MAP)

# --- Recover what the money columns still know ---------------------------
q0 = f["Quantity"] == 0
d1 = f["DiscountPercent"] > 1
f.loc[q0, "Quantity"] = (f["SalesAmount"]
                         / (f["UnitPrice"] * (1 - f["DiscountPercent"]))
                        ).round().loc[q0].astype("Int64")
f.loc[d1, "DiscountPercent"] = (1 - f["SalesAmount"]
                                / (f["Quantity"].astype(float) * f["UnitPrice"])
                               ).round(2).loc[d1]
f["RecoveredField"] = np.select([q0, d1], ["Quantity", "DiscountPercent"], default=None)
print(f"recovered {int(q0.sum() + d1.sum())} rows "
      f"({int(q0.sum())} quantity, {int(d1.sum())} discount)")

# --- Exclusions: counted and reported, never silent ----------------------
before = len(f)
f = f[(f["Quantity"] > 0) & f["DiscountPercent"].between(0, 1)].copy()
print(f"excluded {before - len(f)} rows ({100*(before-len(f))/before:.3f}%) "
      f"- negative-quantity credit notes")

# --- Orphan keys -> Unknown members, never dropped -----------------------
for key, dim in [("CustomerID", c), ("ProductID", p), ("RegionID", region)]:
    f.loc[~f[key].isin(set(dim[key])), key] = "UNKNOWN"

# --- Derived columns ------------------------------------------------------
f["LossFlag"]     = f["ProfitAmount"] < 0
f["DiscountBand"] = pd.cut(f["DiscountPercent"], [-0.001, 0.001, 0.10, 0.20, 0.30, 0.401],
                           labels=["0%","0-10%","10-20%","20-30%","30-40%"])
f["LineMarginPct"] = (100 * f["ProfitAmount"] / f["SalesAmount"]).round(2)

# Flag BOTH money rules - correct neither. Tolerance 0.05, not equality:
# money is stored to 2dp and floats are inexact, so == flags thousands of
# rows that are fine and you stop trusting your own check.
f["FailsPricingRule"] = (
    (f["SalesAmount"] - (f["Quantity"] * f["UnitPrice"] * (1 - f["DiscountPercent"])).round(2))
    .abs() > 0.05)
f["FailsProfitRule"] = (
    ((f["SalesAmount"] - f["CostAmount"]) - f["ProfitAmount"]).abs() > 0.05)

print(f"clean fact: {len(f):,} rows   "
      f"pricing breaches {int(f.FailsPricingRule.sum())}   "
      f"profit breaches {int(f.FailsProfitRule.sum())}")

## 6. The two populations

Cancelled orders carry revenue that never happened, and mixing them into money metrics is
the easiest way to get this wrong. It is what the first version of this analysis did,
overstating revenue by ₹14.5 Cr.

In [ ]:
net    = f[f["OrderStatus"] == "Completed"].copy()    # all money metrics
placed = f[f["OrderStatus"] != "Cancelled"].copy()    # return rate, activity

print((f.groupby("OrderStatus")
        .agg(Lines=("SalesAmount","size"),
             Revenue_Cr=("SalesAmount", lambda s: round(s.sum()/1e7, 2)),
             Profit_Cr=("ProfitAmount", lambda s: round(s.sum()/1e7, 2)))).to_string())

print(f"\nnet {len(net):,} lines   placed {len(placed):,} lines   all {len(f):,}")

In [ ]:
# Headline numbers - these must match sql/04_analysis.sql exactly
rev, prof = net.SalesAmount.sum(), net.ProfitAmount.sum()
print(f"Revenue          Rs {rev/1e7:,.2f} Cr      (expect 176.85)")
print(f"Profit           Rs {prof/1e7:,.2f} Cr      (expect  24.17)")
print(f"Margin                {100*prof/rev:,.2f}%       (expect  13.67)")
print(f"Orders              {net.OrderID.nunique():,}          (expect 68,817)")
print(f"AOV              Rs {rev/net.OrderID.nunique():,.0f}       (expect 25,698)")
print(f"Loss-line rate        {100*net.LossFlag.mean():.1f}%        (expect  11.1)")
print(f"Revenue at risk       {100*net.loc[net.LossFlag,'SalesAmount'].sum()/rev:.1f}%        (expect  18.4)")
print(f"Loss value       Rs {-net.loc[net.LossFlag,'ProfitAmount'].sum()/1e7:,.2f} Cr      (expect   2.59)")

## 7. Validate the clean output

Re-run the detection from section 2. Every count should now be zero or explained.

In [ ]:
checks = {
    "rows":                       (len(f), 157108),
    "unparsed dates":             (int(f.OrderDate.isna().sum()), 0),
    "distinct OrderStatus":       (f.OrderStatus.nunique(), 3),
    "distinct Gender":            (c.Gender.nunique(), 2),
    "distinct Category":          (p.Category.nunique(), 5),
    "duplicate ProductID":        (len(p) - p.ProductID.nunique(), 0),
    "duplicate CustomerID":       (len(c) - c.CustomerID.nunique(), 0),
    "orphan ProductID":           (int((~f.ProductID.isin(set(p.ProductID) | {"UNKNOWN"})).sum()), 0),
    "AgeGroup wrongly '65+'":     (int(((c.AgeGroup == "65+") & c.DateOfBirth.isna()).sum()), 0),
    "untrimmed Category":         (int((p.Category.dropna() != p.Category.dropna().str.strip()).sum()), 0),
}
ok = 0
for k, (got, want) in checks.items():
    good = got == want
    ok += good
    print(f"  {'PASS' if good else '** FAIL **':<12}{k:<26}{got:>8}  expected {want}")
print(f"\n{ok}/{len(checks)} checks passed")

## 8. Write out

Two destinations: the CSVs always, SQL Server if it is reachable.

In [ ]:
# Unknown members so the orphan rows have somewhere to point
p_out = pd.concat([p, pd.DataFrame([{"ProductID":"UNKNOWN","ProductName":"Unknown Product",
                   "Category":"Unknown","SubCategory":"Unknown"}])], ignore_index=True)
c_out = pd.concat([c, pd.DataFrame([{"CustomerID":"UNKNOWN","CustomerName":"Unknown Customer",
                   "Segment":"Unknown","AgeGroup":"Unknown"}])], ignore_index=True)
r_out = pd.concat([region, pd.DataFrame([{"RegionID":"UNKNOWN","Region":"Unknown",
                   "Zone":"Unknown","State":"Unknown","City":"Unknown","Country":"Unknown"}])],
                  ignore_index=True)

TABLES = {"fact_sales": f, "dim_product": p_out, "dim_customer": c_out,
          "dim_region": r_out, "dim_channel": chan, "dim_date": date,
          "dim_userregion": usrg}

for name, df in TABLES.items():
    df.to_csv(CLEAN / f"{name}.csv", index=False, encoding="utf-8")
    print(f"  {name:<16} {len(df):>8,} rows -> {name}.csv")

In [ ]:
# Push to SQL Server so sql/04_analysis.sql reads the same tables.
# Wrapped so a connection failure does not break the notebook - the CSVs
# above are already written either way.
SERVER = r"LOCALHOST\SQLEXPRESS"     # <-- your server name
DB     = "ContosoRetail"

try:
    import sqlalchemy as sa
    eng = sa.create_engine(
        f"mssql+pyodbc://@{SERVER}/{DB}"
        "?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes")
    with eng.begin() as conn:
        for name, df in TABLES.items():
            df.to_sql(name, conn, if_exists="replace", index=False, chunksize=10_000)
            print(f"  wrote {name:<16} {len(df):>8,} rows")
    print("\nSQL Server updated - sql/04_analysis.sql can run now.")
except Exception as e:
    print(f"SQL Server write skipped: {type(e).__name__}: {e}")
    print("The clean CSVs are written regardless. Fix SERVER above, or point")
    print("sql/04_analysis.sql reads data/clean/ instead.")

## 9. The contract with the analysis

Everything above produces columns and `sql/04_analysis.sql` consumes them. Nothing
checked that the two agreed, and for a while they did not. The SQL asked for `IsNetSale`,
a column the retired SQL cleaning path used to create and this notebook never did. Two
queries failed with "Invalid column name" and the channel analysis could not be produced
at all.

The failure surfaced two steps downstream of its cause. This cell moves it back here,
where the columns are written.

It parses the analysis file and compares every name it reads against every column written
here. No database, no query, no connection, just text.

The specific bug took four lines to fix. This check is the fix for the class of bug, so a
future architecture change gets caught at the source instead of surfacing as an error two
steps away.

In [ ]:
# Parse the analysis SQL and prove every column it reads is one I write.
import re
from pathlib import Path

# The analysis file sits in sql/ beside this notebook, or flat alongside it in
# the 'Order to Decode' copy. Check both layouts rather than hard-coding one.
HERE = Path.cwd()
CANDIDATES = [HERE.parent / "sql" / "sql/04_analysis.sql",
              HERE / "04_Analysis.sql",
              HERE / "sql/04_analysis.sql",
              HERE.parent / "04_Analysis.sql"]
sql_path = next((p for p in CANDIDATES if p.exists()), None)

# T-SQL keywords and functions. Anything here is the language, not my data.
# If this check ever flags a name that belongs in this list, add it here -
# that is a gap in the list, not a missing column.
TSQL = set("""
select from where group by order having join left right inner outer full cross on
as and or not in is null distinct count sum avg min max cast convert try_cast
decimal float int bigint nvarchar varchar char bit date datetime case when then
else end with union all over partition row_number rank dense_rank lag lead ntile
datefromparts year month day datediff dateadd eomonth getdate isnull coalesce
nullif abs round floor ceiling len ltrim rtrim trim upper lower concat concat_ws
substring charindex patindex replace create alter view drop table if exists go
desc asc top offset fetch next rows range between unbounded preceding following
current row into values set declare begin return use apply percentile_cont
""".split())

if sql_path is None:
    print("Contract check SKIPPED - analysis SQL not found. Looked in:")
    for p in CANDIDATES:
        print(f"    {p}")
else:
    sql = sql_path.read_text(encoding="utf-8")

    # Strip block comments, line comments, then string literals - in that
    # order. Without this the commentary and values like 'Completed' parse as
    # identifiers and the check cries wolf.
    b = re.sub(r"/\*.*?\*/", " ", sql, flags=re.S)
    b = re.sub(r"--[^\n]*", " ", b)
    b = re.sub(r"'[^']*'", " ", b)

    # What I write out.
    produced = {col.lower() for df in TABLES.values() for col in df.columns}

    # What the query supplies for itself: column aliases, CTE names, table
    # aliases. Not my responsibility to provide.
    derived = {m.lower() for m in re.findall(r"\bAS\s+([A-Za-z_]\w+)", b)}
    ctes = {m.lower() for m in
            re.findall(r"(?:\bWITH|,)\s*([A-Za-z_]\w*)\s+AS\s*\(", b, re.I)}
    aliases = {m.lower() for m in
               re.findall(r"\b([A-Za-z_]\w{0,20})\s*\.\s*[A-Za-z_]", b)}
    objects = ({n.lower() for n in TABLES}
               | {"sales_net", "sales_placed", "stg_fact_sales", "contosoretail"})

    known = produced | derived | ctes | aliases | objects | TSQL
    seen = {t.lower() for t in re.findall(r"\b[A-Za-z_]\w+\b", b)}
    orphans = sorted(seen - known)

    print(f"Contract check: {sql_path.name}")
    print(f"  columns I write          {len(produced):>4}")
    print(f"  names the query defines  {len(derived | ctes | aliases):>4}")
    print(f"  identifiers scanned      {len(seen):>4}")

    if orphans:
        print(f"\n  ** {len(orphans)} identifier(s) that nothing here supplies:")
        for o in orphans:
            hits = [i + 1 for i, ln in enumerate(sql.splitlines())
                    if re.search(rf"\b{re.escape(o)}\b", ln, re.I)]
            print(f"       {o:<22} line(s) {hits[:8]}")
        raise AssertionError(
            f"{sql_path.name} reads {len(orphans)} name(s) with no producer: "
            f"{orphans}.\nEither this notebook must create them, or the SQL must "
            f"stop asking.\nDo not resolve it by adding a column that repeats a "
            f"rule the views already own - that is how IsNetSale happened.\n"
            f"If one of these is a T-SQL keyword or function, add it to TSQL above.")

    print("\n  PASS - every name the analysis reads is either a column I write")
    print("         or one the query defines for itself.")


## Done

```
data/clean/*.csv                seven clean tables
SQL Server dim_* / fact_sales   same, if the write above succeeded
```

Next: `sql/04_analysis.sql` for Q1-Q15, then `notebooks/05_statistics.ipynb`.

Section 9 has already confirmed the analysis asks for nothing this notebook does not
write, so the only failures left downstream are T-SQL ones.

If the headline numbers in section 6 do not match, stop there. Either a map is missing a
value or an exclusion ran that should not have, and it is much harder to find once it is
in the analysis.